# HW2 Playground

Fill in TODOs as you work through the assignment.
Implement the required sections in `model.py`, and use this notebook to orchestrate and run your solution.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from hw2_loader import HW2DataLoader
from model import GradientBoostingModel

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


Failed to read module file 'C:\Users\Jonathan Xu\AppData\Local\Programs\Python\Python311\Lib\re\_casefix.py' for module 're._casefix': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\Jonathan Xu\Documents\CS_CLASSES\CSCI1851\homework-2-generoussidewalk\homework2\.hw2\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Jonathan Xu\Documents\CS_CLASSES\CSCI1851\homework-2-generoussidewalk\homework2\.hw2\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Jonathan Xu\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1

In [ ]:
# TODO: Load both datasets
loader = HW2DataLoader()

# Heart disease dataset
heart_path = Path('data/heart.csv')
X_heart, y_heart = loader.get_heart_disease_data(csv_path=heart_path)
print(X_heart.shape, y_heart.value_counts().to_dict())

full_heart = pd.concat([X_heart, y_heart], axis=1)
print(f"Duplicate rows before dedup: {full_heart.duplicated().sum()}")
full_heart= full_heart.drop_duplicates().reset_index(drop=True)
X_heart =full_heart.drop(columns=[y_heart.name])
y_heart = full_heart[y_heart.name]
print(f"Shape after dedup: {X_heart.shape}")

# Cancer genomics dataset
cancer_path = Path('data/cancer_genomics.csv')
labels_path = Path('data/labels_cancer_genomics.csv')
X_cancer, y_cancer = loader.get_cancer_genomics_data(
    csv_path=cancer_path, labels_path=labels_path
)
print(X_cancer.shape, y_cancer.value_counts().to_dict())

Successfully loaded heart disease data with 1025 rows
(1025, 13) {1: 526, 0: 499}
Duplicate rows before dedup: 723
Shape after dedup: (302, 13)
(801, 5479) {'BRCA': 300, 'KIRC': 146, 'LUAD': 141, 'PRAD': 136, 'COAD': 78}


In [ ]:
# Train and evaluate heart dataset WITHOUT scaler
model_heart = GradientBoostingModel(task='classification', max_depth=3, learning_rate=0.1, n_estimators=100, use_scaler=False)
X_train_h ,X_test_h, y_train_h, y_test_h = model_heart.train_test_split(X_heart, y_heart)
model_heart.fit(X_train_h, y_train_h)
metrics_no_scaler = model_heart.evaluate(X_test_h, y_test_h)
print("Heart (no scaler):", metrics_no_scaler)

# Train and evaluate heart dataset WITH scaler
model_heart_scaled = GradientBoostingModel(task='classification', max_depth=3, learning_rate=0.1, n_estimators=100, use_scaler=True)
model_heart_scaled.train_test_split(X_heart, y_heart) 
model_heart_scaled.fit(X_train_h, y_train_h)
metrics_scaler = model_heart_scaled.evaluate(X_test_h, y_test_h)
print("Heart (scaler):   ", metrics_scaler)

Model trained with 100 estimators


In [ ]:
# Cross-validation on heart dataset (no scaler vs scaler)
for use_scaler in [False, True]:
    label="scaler" if use_scaler else "no scaler"
    m = GradientBoostingModel(task='classification', max_depth=3, learning_rate=0.1, n_estimators=100, use_scaler=use_scaler)
    cv_results = m.cross_validate(X_heart, y_heart, cv=5)
    print(f"\nHeart CV ({label})")
    for metric, result in cv_results.items():
        print(f"  {metric}: mean={result['mean']:.4f}, std={result['std']:.4f}")

accuracy: mean=0.8045, std=0.0575
precision_weighted: mean=0.8051, std=0.0579
recall_weighted: mean=0.8045, std=0.0575
f1_weighted: mean=0.8034, std=0.0582
roc_auc_ovr_weighted: mean=0.8815, std=0.0479


In [ ]:
# Experiment with different depths and ensemble sizes (heart)
results_heart =[]
for depth in [1, 3, 5]:
    for n_est in [50, 100, 200]:
        m =GradientBoostingModel(task='classification', max_depth=depth, learning_rate=0.1, n_estimators=n_est)
        m.train_test_split(X_heart, y_heart)
        m.fit(X_train_h, y_train_h, verbose=False)
        met = m.evaluate(X_test_h, y_test_h)
        met['max_depth'] = depth
        met['n_estimators'] = n_est
        results_heart.append(met)

results_heart_df = pd.DataFrame(results_heart)
print(results_heart_df.to_string(index=False))

In [ ]:
# Hyperparameter tuning (heart)
param_grid = {
    'max_depth': [2, 3, 5],
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
}
tuning_heart= model_heart.tune_hyperparameters(X_heart, y_heart, param_grid, cv=3)
print("Best params:",tuning_heart['best_params'])
print("Best ROC AUC:", tuning_heart['best_score'])

{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 50}
0.8995364270726589


In [ ]:
# Feature importance (heart)
heart_importance = model_heart.get_feature_importance(plot=True)
print(heart_importance.head(10))

In [ ]:
# Train and evaluate cancer dataset WITHOUT scaler
cancer_model = GradientBoostingModel(task='classification', max_depth=3, learning_rate=0.1, n_estimators=100, use_scaler=False)
X_train_c, X_test_c, y_train_c, y_test_c = cancer_model.train_test_split(X_cancer, y_cancer)
cancer_model.fit(X_train_c, y_train_c)
cancer_metrics_no_scaler = cancer_model.evaluate(X_test_c, y_test_c)
print("Cancer (no scaler):", cancer_metrics_no_scaler)

# Train and evaluate cancer dataset WITH scaler
cancer_model_scaled= GradientBoostingModel(task='classification', max_depth=3, learning_rate=0.1, n_estimators=100, use_scaler=True)
cancer_model_scaled.train_test_split(X_cancer, y_cancer)
cancer_model_scaled.fit(X_train_c, y_train_c)
cancer_metrics_scaler = cancer_model_scaled.evaluate(X_test_c, y_test_c)
print("Cancer (scaler):   ", cancer_metrics_scaler)

In [ ]:
# Cross-validation on cancer dataset (no scaler vs scaler)
for use_scaler in [False, True]:
    label = "scaler" if use_scaler else "no scaler"
    m = GradientBoostingModel(task='classification', max_depth=3, learning_rate=0.1, n_estimators=100, use_scaler=use_scaler)
    cv_results_c = m.cross_validate(X_cancer, y_cancer, cv=5)
    print(f"\nCancer CV ({label})")
    for metric, result in cv_results_c.items():
        print(f"  {metric}: mean={result['mean']:.4f}, std={result['std']:.4f}")

In [ ]:
# Experiment with different depths and ensemble sizes (cancer)
results_cancer = []
for depth in [1, 3, 5]:
    for n_est in [50, 100, 200]:
        m = GradientBoostingModel(task='classification', max_depth=depth, learning_rate=0.1, n_estimators=n_est)
        m.train_test_split(X_cancer, y_cancer)
        m.fit(X_train_c, y_train_c, verbose=False)
        met = m.evaluate(X_test_c, y_test_c)
        met['max_depth'] = depth
        met['n_estimators'] = n_est
        results_cancer.append(met)

results_cancer_df = pd.DataFrame(results_cancer)
print(results_cancer_df.to_string(index=False))

In [ ]:
# Hyperparameter tuning (cancer). weighted scoring for multi-class
param_grid_c = {
    'max_depth': [2, 3, 5],
    'n_estimators': [50,100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
}
tuning_cancer = cancer_model.tune_hyperparameters(
    X_cancer, y_cancer, param_grid_c, cv=3, scoring='roc_auc_ovr_weighted'
)
print("Best params:", tuning_cancer['best_params'])
print("Best ROC AUC (weighted):", tuning_cancer['best_score'])

In [ ]:
# Feature importance (cancer)
cancer_importance = cancer_model.get_feature_importance(plot=True, top_n=20)
print(cancer_importance.head(20))